# Aula 20 · Lakeflow Jobs em produção e melhores práticas

## O que é
O conjunto de práticas para levar um job do "funciona na minha máquina" para **produção**: seguro, barato e fácil de manter.

## Por que importa
O que funciona em desenvolvimento nem sempre é seguro, barato e sustentável em produção, e esse cuidado é **cobrado em processos seletivos**.

## Como aplicar: checklist de prontidão para produção
Revise o job `growdev_pipeline_emprestimos` item por item.

| # | Prática | Por quê | No nosso job |
|---|---|---|---|
| 1 | **Compute serverless ou cluster de job.** Nunca um cluster interativo | O interativo fica ligado (e cobrando) e é compartilhado com pessoas | ✅ Serverless |
| 2 | **Orquestração modular** | Tarefas pequenas, com uma responsabilidade cada, são fáceis de testar e reparar | ✅ Um notebook por etapa |
| 3 | **Parametrização** | O mesmo job serve para dev e prod, só trocando parâmetros | ✅ `catalogo`, `limite_nulos_pct`, `limite_risco_pct` |
| 4 | **Idempotência** | Retentativa e reparo sem duplicar dado | ✅ Bronze incremental, `MERGE`, `overwrite` |
| 5 | **Retentativas, timeouts e alertas** | Falha transitória se resolve sozinha; falha real avisa o time | ✅ Aula 18 |
| 6 | **Permissões** | Só quem deve pode editar ou executar | ⬜ Configure em **Permissions** |
| 7 | **Run as service principal** | O job não pode parar quando a pessoa que o criou sai da empresa | ⬜ Em empresas, use um *service principal* |
| 8 | **Uma execução por vez** | Evita duas execuções gravando ao mesmo tempo | ✅ *Max concurrent runs* = 1 |
| 9 | **Versionamento e implantação como código** | Histórico de mudanças, revisão por pares, deploy repetível em dev/prod | ✅ Declarative Automation Bundles (abaixo) |

### 6. Permissões
No job → **Permissions**:
- `CAN_VIEW`: quem só acompanha (ex.: analistas)
- `CAN_MANAGE_RUN`: quem pode executar e reparar (ex.: time de sustentação)
- `CAN_MANAGE`: quem pode editar (ex.: engenheiros donos do job)

### 7. Run as
No job → painel da direita → **Run as**. Em uma empresa, troque o seu usuário por um **service principal**: uma "conta de robô" que não sai de férias nem muda de time.

> No plano gratuito o job roda como o seu usuário. Mostramos onde fica a configuração.

### 9. Job como código: Declarative Automation Bundles
O repositório já tem a **versão final do job como código**:

```
databricks.yml                                   ← configuração do bundle
resources/growdev_pipeline_emprestimos.job.yml   ← o job completo deste módulo
resources/growdev_ingestao.job.yml               ← Agrupamento 7: módulo de ingestão
resources/growdev_transformacao.job.yml          ← Agrupamento 7: módulo de transformação
resources/growdev_orquestrador.job.yml           ← Agrupamento 7: orquestrador
resources/growdev_ingestao_gerenciada.pipeline.yml ← Agrupamento 7: pipeline de ingestão
```

Abra `resources/growdev_pipeline_emprestimos.job.yml` e reconheça tudo o que fizemos pela interface: parâmetros, For each, If/else, Run if, retentativas, timeout, alerta de duração, notificações e gatilho por arquivo.

Para implantar pelo terminal, com a [Databricks CLI](https://docs.databricks.com/dev-tools/cli/install.html) instalada:

```bash
databricks bundle validate      # confere se o arquivo está correto
databricks bundle deploy        # cria/atualiza o job no workspace (target dev)
databricks bundle run growdev_pipeline_emprestimos
```

No target `dev`, o bundle coloca o prefixo `[dev seu_usuario]` no nome do job e **pausa os gatilhos**, para não atrapalhar a produção.

## Próximos passos de estudo e carreira
- Certificação **Databricks Certified Data Engineer Associate**
- Formação oficial **Deploy Workloads with Lakeflow Jobs** (Databricks Academy)
- Publique este projeto no seu GitHub: ele é uma peça de portfólio

## Checklist da aula
- [ ] Revisei o job contra os 9 itens
- [ ] Configurei permissões no job
- [ ] Sei o que é "Run as" e por que usar service principal
- [ ] Li o job como código no bundle e sei para que servem `validate`, `deploy` e `run`